In [4]:
import pandas as pd
from pathlib import Path
import duckdb
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

try:
    # if running as a .py, __file__ exists
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    # if running in a notebook, __file__ does NOT exist
    BASE_DIR = Path.cwd().parents[1] 

# path to database
DATA_DIR = BASE_DIR / "data" / "train_delay_processed.duckdb"
print(DATA_DIR)

/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project/data/train_delay_processed.duckdb


In [5]:
# establish SQL connection to database and load into dataframe
with duckdb.connect(DATA_DIR) as con:
    df = con.execute("SELECT * FROM train_delay_processed").df()

In [ ]:
# sort by time
df = df.sort_values("time_current_real")

# Get unique journeys
journeys = df["journey_id"].unique()

# Split journeys
tscv = TimeSeriesSplit(n_splits=5)

In [10]:
le_station = LabelEncoder()
df["station_current_enc"] = le_station.fit_transform(df["station_current"])


for i, (train_idx, test_idx) in enumerate(tscv.split(journeys)):
    train_journeys = journeys[train_idx]
    test_journeys = journeys[test_idx]
    
    train_df = df[df["journey_id"].isin(train_journeys)]
    test_df = df[df["journey_id"].isin(test_journeys)]
    
    # Features and target
    X_train = train_df[["station_current_enc", "stops_total"]]
    y_train = train_df["delay"]
    
    X_test = test_df[["station_current_enc", "stops_total"]]
    y_test = test_df["delay"]
    
    # Train Random Forest
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Predict and evaluate
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"Fold {i}: Test RMSE = {rmse:.2f}")


std_delay = df['delay'].std()
print(f"Delay standard deviation: {std_delay:.2f}")

Fold 0: Test RMSE = 19.64
Fold 1: Test RMSE = 21.07
Fold 2: Test RMSE = 19.31
Fold 3: Test RMSE = 21.46
Fold 4: Test RMSE = 15.28
Delay standard deviation: 17.28
